
# GVH Diagonal Cubic 0.3.2.7.3.7.3.3.1.6.1
## Covariant \(C_N^{\rm loc}\) Metric Functional Jet and Connection-Variation Closure

**Auteur :** Charlemagne O Laurince  
**Branche :** `0.2C1_prediction_foundations`  
**Prédécesseur direct :** `0.3.2.7.3.7.3.3.1.6`  
**Traceabilité :** `REDERIVED / ACTUAL-GVH-QJU / NO-MODEL-MODIFICATION`

---

# Mission unique

Fermer le bloc laissé ouvert par `.1.6` :

\[
\boxed{
\frac{\delta}{\delta h_{mn}}
\int d^3x\,\sqrt h\,N\,C_N^{\rm loc}
}
\]

sans :

- modifier l'action ;
- ajouter un mécanisme physique ;
- supposer la fermeture ADM ;
- injecter \(D[\beta]\) dans le résultat ;
- ouvrir une branche RACC, PPN ou dispersion.

Le calcul conserve explicitement :

\[
v^i=h^{ij}v_j,
\]

\[
q_{ij}\equiv D_i v_j,
\]

\[
\delta q_{ij}\big|_{v}
=
-\delta\Gamma^k{}_{ij}v_k,
\]

\[
\delta\Gamma^k{}_{ij}
=
\frac12h^{kl}
\left(
D_i\delta h_{jl}
+D_j\delta h_{il}
-D_l\delta h_{ij}
\right).
\]

Le calcul utilise des **coordonnées normales spatiales au point considéré** :

\[
h_{ij}=\delta_{ij},
\qquad
\Gamma^k{}_{ij}=0,
\]

mais **ne pose jamais**

\[
\delta\Gamma^k{}_{ij}=0.
\]

C'est précisément la variation de connexion qui doit être restaurée.

À la fin de cette étape, si tous les gates passent :

\[
\boxed{
\texttt{CNLOC\_COVARIANT\_METRIC\_FUNCTIONAL\_JET\_EXPLICIT=True}
}
\]

mais le crochet HH lui-même reste réservé au réassemblage immédiat suivant.

\[
\boxed{\mathrm{DISPERSION\_READY=False}}
\]


In [ ]:

from __future__ import annotations
import sympy as sp
import json, sys
from pathlib import Path

print("GVH 0.3.2.7.3.7.3.3.1.6.1")
print("Python:", sys.version.split()[0])
print("SymPy:", sp.__version__)



# 1 — Pourquoi les coordonnées normales suffisent

Le jet fonctionnel métrique est tensoriel.

Au voisinage d'un point \(x_0\), on peut choisir :

\[
h_{ij}(x_0)=\delta_{ij},
\qquad
\Gamma^k{}_{ij}(x_0)=0,
\]

tout en gardant :

\[
D_l\delta h_{ij}(x_0)\neq0.
\]

Ainsi :

- la **variation algébrique** des contractions métriques peut être calculée autour de \(h=I\) ;
- la **variation différentielle** de \(D_i v_j\) est portée séparément par \(\delta\Gamma\).

Cette séparation évite de construire un \(Q^{-1}(h)\) symbolique gigantesque tout en conservant le jet covariant exact au premier ordre.


In [ ]:

# Variation métrique symétrique indépendante.
e11,e22,e33,e12,e13,e23,eps = sp.symbols(
    "e11 e22 e33 e12 e13 e23 eps",
    real=True
)

dH = sp.Matrix([
    [e11,e12,e13],
    [e12,e22,e23],
    [e13,e23,e33],
])

I3m = sp.eye(3)

# Exact au premier ordre autour de h_ij = delta_ij:
# delta h^{ij} = - delta h_ij.
hinv_eps = I3m - eps*dH

trace_dh = sp.trace(dH)

assert trace_dh == e11+e22+e33

NORMAL_COORDINATE_FIRST_VARIATION_SETUP = True
print(
    "NORMAL_COORDINATE_FIRST_VARIATION_SETUP =",
    NORMAL_COORDINATE_FIRST_VARIATION_SETUP
)



# 2 — Reconstruction covariante des invariants locaux

On reprend exactement les variables amont :

\[
V^A=
(K_{11},K_{22},K_{33},K_{12},K_{13},K_{23},S,W_1,W_2,W_3).
\]

Les objets covariants sont :

\[
A=-S-v^ia_i,
\]

\[
B_i=s\,a_i+W_i-K_i{}^jv_j,
\]

\[
C_i=-D_is-K_i{}^jv_j,
\]

\[
D_{ij}=D_iv_j+sK_{ij}.
\]

Contrairement à la représentation orthonormée précédente, les contractions sont ici écrites avec \(h^{ij}\) avant la variation.


In [ ]:

c1,c2,c3,c4,s = sp.symbols(
    "c1 c2 c3 c4 s",
    real=True
)

v = sp.Matrix(sp.symbols("v1:4", real=True))
avec = sp.Matrix(sp.symbols("a1:4", real=True))
g = sp.Matrix(sp.symbols("g1:4", real=True))

qsyms = sp.symbols(
    "q11 q12 q13 q21 q22 q23 q31 q32 q33",
    real=True
)
q = sp.Matrix(3,3,qsyms)

K11,K22,K33,K12,K13,K23,S,W1,W2,W3 = sp.symbols(
    "K11 K22 K33 K12 K13 K23 S W1 W2 W3",
    real=True
)

vel = sp.Matrix([
    K11,K22,K33,K12,K13,K23,
    S,W1,W2,W3
])

K = sp.Matrix([
    [K11,K12,K13],
    [K12,K22,K23],
    [K13,K23,K33],
])

W = sp.Matrix([W1,W2,W3])

v_up_eps = hinv_eps*v
Kmix_eps = K*hinv_eps

A_eps = sp.expand(
    -S-(v_up_eps.T*avec)[0]
)

B_eps = sp.expand(
    s*avec+W-Kmix_eps*v
)

C_eps = sp.expand(
    -g-Kmix_eps*v
)

D = q+s*K

def cov_dot_cov_eps(x,y):
    return (x.T*hinv_eps*y)[0]

Dnorm_eps = sp.trace(
    hinv_eps*D*hinv_eps*D.T
)

Dswap_eps = sp.trace(
    hinv_eps*D*hinv_eps*D
)

theta_eps = sp.expand(
    -A_eps+sp.trace(hinv_eps*D)
)

I1_eps = (
    A_eps**2
    -cov_dot_cov_eps(B_eps,B_eps)
    -cov_dot_cov_eps(C_eps,C_eps)
    +Dnorm_eps
)

I3_eps = (
    A_eps**2
    -2*cov_dot_cov_eps(B_eps,C_eps)
    +Dswap_eps
)

alpha_eps = (
    s*A_eps+(v_up_eps.T*C_eps)[0]
)

beta_cov_eps = (
    s*B_eps+D.T*v_up_eps
)

acc2_eps = (
    -alpha_eps**2
    +cov_dot_cov_eps(beta_cov_eps,beta_cov_eps)
)

Lu_eps = (
    -c1*I1_eps
    -c2*theta_eps**2
    -c3*I3_eps
    +c4*acc2_eps
)

Knorm_eps = sp.trace(
    hinv_eps*K*hinv_eps*K.T
)

Ktrace_eps = sp.trace(
    hinv_eps*K
)

LEH_eps = (
    Knorm_eps-Ktrace_eps**2
)

Ltot_eps = sp.expand(
    Lu_eps+LEH_eps
)

COVARIANT_LOCAL_INVARIANTS_RECONSTRUCTED = True

print(
    "COVARIANT_LOCAL_INVARIANTS_RECONSTRUCTED =",
    COVARIANT_LOCAL_INVARIANTS_RECONSTRUCTED
)



# 3 — Contrôle de réduction vers la représentation amont

À :

\[
\varepsilon=0
\]

on doit retrouver exactement la représentation orthonormée utilisée dans `.1.4` et `.1.6`.


In [ ]:

A0 = sp.expand(-S-v.dot(avec))
B0 = sp.expand(s*avec+W-K*v)
C0 = sp.expand(-g-K*v)
D0 = sp.expand(q+s*K)

I1_0 = sp.expand(
    A0**2
    -B0.dot(B0)
    -C0.dot(C0)
    +sum(
        D0[i,j]**2
        for i in range(3)
        for j in range(3)
    )
)

theta0 = sp.expand(
    -A0+sp.trace(D0)
)

I3_0 = sp.expand(
    A0**2
    -2*B0.dot(C0)
    +sum(
        D0[i,j]*D0[j,i]
        for i in range(3)
        for j in range(3)
    )
)

alpha0 = sp.expand(
    s*A0+v.dot(C0)
)

beta0 = sp.expand(
    s*B0+D0.T*v
)

acc2_0 = sp.expand(
    -alpha0**2+beta0.dot(beta0)
)

Lu0 = sp.expand(
    -c1*I1_0
    -c2*theta0**2
    -c3*I3_0
    +c4*acc2_0
)

LEH0 = sp.expand(
    sum(
        K[i,j]**2
        for i in range(3)
        for j in range(3)
    )
    -sp.trace(K)**2
)

Ltot0 = sp.expand(
    Lu0+LEH0
)

assert sp.expand(
    Ltot_eps.subs(eps,0)-Ltot0
) == 0

ORTHONORMAL_UPSTREAM_REDUCTION_EXACT = True

print(
    "ORTHONORMAL_UPSTREAM_REDUCTION_EXACT =",
    ORTHONORMAL_UPSTREAM_REDUCTION_EXACT
)



# 4 — Première variation algébrique métrique

On définit :

\[
\delta_h^{\rm alg}L
=
\left.
\frac{d}{d\varepsilon}
L[h+\varepsilon\delta h]
\right|_{\varepsilon=0}
\]

à :

\[
q_{ij}=D_i v_j
\]

tenu momentanément fixe.

La variation de connexion sera ajoutée séparément, afin d'éviter tout double comptage.


In [ ]:

dL_alg = sp.expand(
    sp.diff(Ltot_eps,eps).subs(eps,0)
)

assert sp.diff(dL_alg,eps) == 0

ALGEBRAIC_METRIC_FIRST_VARIATION_EXPLICIT = True

print(
    "ALGEBRAIC_METRIC_FIRST_VARIATION_EXPLICIT =",
    ALGEBRAIC_METRIC_FIRST_VARIATION_EXPLICIT
)
print(
    "first-variation expression length =",
    len(str(dL_alg))
)



# 5 — Réponses \(Q_{,h}\), \(J_{0,h}\), \(U_{0,h}\)

La décomposition locale est :

\[
L
=
\frac12V^TQV+J^TV+U.
\]

Puisque variation métrique et dérivation par rapport aux vitesses commutent :

\[
\delta Q
=
\frac{\partial^2(\delta L)}
{\partial V\,\partial V},
\]

\[
\delta J_0
=
\left.
\frac{\partial(\delta L)}{\partial V}
\right|_{V=0,a=0},
\]

\[
\delta U_0
=
\left.
\delta L
\right|_{V=0,a=0}.
\]

Cela évite toute dérivation de \(Q^{-1}\).


In [ ]:

zero_vel = {x:0 for x in vel}
zero_a = {x:0 for x in avec}

Q0 = sp.hessian(
    Ltot0,
    list(vel)
)

J0 = sp.Matrix([
    sp.diff(Ltot0,x).subs(zero_vel)
    for x in vel
]).subs(zero_a)

U0 = sp.expand(
    Ltot0
    .subs(zero_vel)
    .subs(zero_a)
)

dQ_alg = sp.hessian(
    dL_alg,
    list(vel)
).subs(zero_a)

dJ_alg = sp.Matrix([
    sp.diff(dL_alg,x)
    .subs(zero_vel)
    .subs(zero_a)
    for x in vel
])

dU_alg = sp.expand(
    dL_alg
    .subs(zero_vel)
    .subs(zero_a)
)

assert Q0 == Q0.T
assert dQ_alg == dQ_alg.T

QJU_METRIC_RESPONSE_EXPLICIT = True

print("Q0 shape =",Q0.shape)
print("dQ_alg shape =",dQ_alg.shape)
print(
    "QJU_METRIC_RESPONSE_EXPLICIT =",
    QJU_METRIC_RESPONSE_EXPLICIT
)



# 6 — Variable de Legendre compacte \(X\)

Sur la branche inversible :

\[
\boxed{
X
=
Q^{-1}(P-J_0)
}
\]

ou de façon équivalente :

\[
\boxed{
QX=P-J_0.
}
\]

Le Hamiltonien local compact est :

\[
\boxed{
C_N^{\rm loc}
=
-\frac12(P-J_0)^TX+U_0.
}
\]

Pour une variation quelconque :

\[
\boxed{
\delta C_N^{\rm loc}
=
(\delta J_0)^TX
+
\frac12X^T(\delta Q)X
+
\delta U_0
-
X^T\delta P.
}
\]


In [ ]:

P = sp.Matrix(
    sp.symbols("P0:10", real=True)
)

X = sp.Matrix(
    sp.symbols("X0:10", real=True)
)

legendre_relation = (
    Q0*X-(P-J0)
)

CNloc_compact = sp.expand(
    -sp.Rational(1,2)*
    ((P-J0).T*X)[0]
    +U0
)

LEGENDRE_COMPACT_REPRESENTATION_READY = True

print(
    "LEGENDRE_COMPACT_REPRESENTATION_READY =",
    LEGENDRE_COMPACT_REPRESENTATION_READY
)
print(
    "Legendre relation retained as Q X = P-J0"
)



# 7 — Variation des moments normalisés à moment canonique fixé

Les dix moments \(P_A\) sont les moments **locaux normalisés**.

Les résultats précédents donnent :

\[
P_s=\frac{p_s}{\sqrt h},
\qquad
P_v^i=\frac{p_v^i}{\sqrt h},
\]

et pour le secteur métrique :

\[
P_{ii}
=
\frac{2\pi^{ii}}{\sqrt h},
\qquad
P_{ij}
=
\frac{4\pi^{ij}}{\sqrt h},
\quad i<j.
\]

À densités canoniques fixées :

\[
\boxed{
\delta P_A
=
-\frac12P_A\,h^{mn}\delta h_{mn}.
}
\]

En coordonnées normales :

\[
\delta P_A
=
-\frac12P_A\,\mathrm{tr}(\delta h).
\]


In [ ]:

deltaP = sp.Matrix([
    -sp.Rational(1,2)*trace_dh*P[i]
    for i in range(10)
])

assert len(deltaP) == 10

NORMALIZED_MOMENT_METRIC_RESPONSE_EXPLICIT = True

print(
    "NORMALIZED_MOMENT_METRIC_RESPONSE_EXPLICIT =",
    NORMALIZED_MOMENT_METRIC_RESPONSE_EXPLICIT
)



# 8 — Variation algébrique de la densité \(\sqrt h\,C_N^{\rm loc}\)

Comme :

\[
\delta\sqrt h
=
\frac12\sqrt h\,h^{mn}\delta h_{mn},
\]

on obtient :

\[
\frac{\delta_{\rm alg}
(\sqrt h C_N^{\rm loc})}{\sqrt h}
=
(\delta J_0)^TX
+
\frac12X^T(\delta Q)X
+
\delta U_0
-
X^T\delta P
+
\frac12C_N^{\rm loc}
h^{mn}\delta h_{mn}.
\]

Cette expression contient :

- les variations de toutes les contractions dans les invariants ;
- \(v^i=h^{ij}v_j\) ;
- \(K_i{}^j\) ;
- les contractions de \(D_i v_j\) ;
- le déterminant \(\sqrt h\) ;
- la normalisation métrique des dix moments \(P_A\).

Elle ne contient pas encore \(\delta\Gamma\), qui est ajouté ensuite.


In [ ]:

d_density_alg_over_sqrth = sp.expand(
    (dJ_alg.T*X)[0]
    +
    sp.Rational(1,2)*
    (X.T*dQ_alg*X)[0]
    +
    dU_alg
    -
    X.dot(deltaP)
    +
    sp.Rational(1,2)*
    trace_dh*CNloc_compact
)

metric_variation_symbols = [
    e11,e22,e33,e12,e13,e23
]

independent_coeffs = [
    sp.expand(
        sp.diff(
            d_density_alg_over_sqrth,
            e
        )
    )
    for e in metric_variation_symbols
]

# Convert from six independent symmetric matrix coordinates to
# tensor coefficients T^{mn} in sum_{mn} T^{mn} delta h_{mn}.
Talg = sp.Matrix([
    [
        independent_coeffs[0],
        sp.Rational(1,2)*independent_coeffs[3],
        sp.Rational(1,2)*independent_coeffs[4],
    ],
    [
        sp.Rational(1,2)*independent_coeffs[3],
        independent_coeffs[1],
        sp.Rational(1,2)*independent_coeffs[5],
    ],
    [
        sp.Rational(1,2)*independent_coeffs[4],
        sp.Rational(1,2)*independent_coeffs[5],
        independent_coeffs[2],
    ],
])

assert Talg == Talg.T

ALGEBRAIC_METRIC_RESPONSE_TENSOR_EXPLICIT = True

print(
    "ALGEBRAIC_METRIC_RESPONSE_TENSOR_EXPLICIT =",
    ALGEBRAIC_METRIC_RESPONSE_TENSOR_EXPLICIT
)
print(
    "Talg component lengths =",
    [
        len(str(Talg[i,j]))
        for i in range(3)
        for j in range(i,3)
    ]
)



# 9 — Réponse au jet \(q_{ij}=D_i v_j\)

Posons :

\[
\boxed{
\mathcal A^{ij}
\equiv
\frac{\partial C_N^{\rm loc}}
{\partial q_{ij}}
}
\]

à \(h\), \(P\), \(s\), \(v_i\) fixés.

Grâce à l'identité de Legendre :

\[
\boxed{
\mathcal A^{ij}
=
J_{0,q_{ij}}^TX
+
\frac12X^TQ_{,q_{ij}}X
+
U_{0,q_{ij}}.
}
\]

Aucune inversion explicite de \(Q\) n'est nécessaire.


In [ ]:

Aq_entries = []

for qij in list(q):
    Q_q = Q0.diff(qij)
    J_q = J0.diff(qij)
    U_q = sp.diff(U0,qij)

    response = sp.expand(
        (J_q.T*X)[0]
        +
        sp.Rational(1,2)*
        (X.T*Q_q*X)[0]
        +
        U_q
    )

    Aq_entries.append(response)

Aq = sp.Matrix(
    3,3,
    Aq_entries
)

QJET_RESPONSE_EXPLICIT = True

print(
    "QJET_RESPONSE_EXPLICIT =",
    QJET_RESPONSE_EXPLICIT
)
print(
    "Aq component lengths =",
    [
        len(str(Aq[i,j]))
        for i in range(3)
        for j in range(3)
    ]
)

# No symmetry is imposed: D_i v_j is not assumed symmetric.
Aq_antisymmetric_part = sp.simplify(
    Aq-Aq.T
)

print(
    "Aq symmetric identically? ",
    Aq_antisymmetric_part == sp.zeros(3)
)



# 10 — Variation de connexion

À \(v_i\) fixé :

\[
\delta q_{ij}
=
-\delta\Gamma^k{}_{ij}v_k.
\]

La contribution au fonctionnel smeared est :

\[
\delta_\Gamma I_N
=
-\int d^3x\,
\sqrt h\,N\,
\mathcal A^{ij}v_k
\delta\Gamma^k{}_{ij}.
\]

En substituant :

\[
\delta\Gamma^k{}_{ij}
=
\frac12h^{kl}
(D_i\delta h_{jl}
+D_j\delta h_{il}
-D_l\delta h_{ij})
\]

et en intégrant par parties, on obtient :

\[
\boxed{
\mathcal K_\Gamma^{mn}[N]
=
\frac14D_i
\left[
N(\mathcal A^{im}v^n+\mathcal A^{in}v^m)
\right]
}
\]

\[
\boxed{
\quad+
\frac14D_j
\left[
N(\mathcal A^{mj}v^n+\mathcal A^{nj}v^m)
\right]
-
\frac14D_l
\left[
N(\mathcal A^{mn}+\mathcal A^{nm})v^l
\right].
}
\]

Aucune symétrie de \(\mathcal A^{ij}\) n'est supposée.


In [ ]:

# Contrôle combinatoire abstrait de la formule d'intégration par parties.
#
# DW[(r,i,j,l)] représente D_r W^{i j l},
# avec W^{i j l}=N A^{ij} v^l.

DW = {}
for r in range(3):
    for i in range(3):
        for j in range(3):
            for l in range(3):
                DW[(r,i,j,l)] = sp.Symbol(
                    f"d{r}W{i}{j}{l}"
                )

def raw_independent_coefficient(a,b):
    # coefficient de la variable symétrique indépendante delta h_ab,
    # a<=b, obtenu directement après IBP.
    total = 0

    for i in range(3):
        for j in range(3):
            for l in range(3):
                # +1/2 D_i W^{i j l} delta h_{j l}
                if (
                    (j==a and l==b)
                    or
                    (a!=b and j==b and l==a)
                ):
                    total += sp.Rational(1,2)*DW[(i,i,j,l)]

                # +1/2 D_j W^{i j l} delta h_{i l}
                if (
                    (i==a and l==b)
                    or
                    (a!=b and i==b and l==a)
                ):
                    total += sp.Rational(1,2)*DW[(j,i,j,l)]

                # -1/2 D_l W^{i j l} delta h_{i j}
                if (
                    (i==a and j==b)
                    or
                    (a!=b and i==b and j==a)
                ):
                    total -= sp.Rational(1,2)*DW[(l,i,j,l)]

    return sp.expand(total)

def K_tensor_component(a,b):
    total = 0

    for i in range(3):
        total += sp.Rational(1,4)*(
            DW[(i,i,a,b)]
            +
            DW[(i,i,b,a)]
        )

    for j in range(3):
        total += sp.Rational(1,4)*(
            DW[(j,a,j,b)]
            +
            DW[(j,b,j,a)]
        )

    for l in range(3):
        total -= sp.Rational(1,4)*(
            DW[(l,a,b,l)]
            +
            DW[(l,b,a,l)]
        )

    return sp.expand(total)

for a in range(3):
    for b in range(a,3):
        raw = raw_independent_coefficient(a,b)
        tensor = K_tensor_component(a,b)

        if a == b:
            assert sp.expand(raw-tensor) == 0
        else:
            # independent off-diagonal variation receives 2*K^{ab}
            assert sp.expand(raw-2*tensor) == 0

CONNECTION_VARIATION_IBP_IDENTITY_VERIFIED = True

print(
    "CONNECTION_VARIATION_IBP_IDENTITY_VERIFIED =",
    CONNECTION_VARIATION_IBP_IDENTITY_VERIFIED
)



# 11 — Jet métrique fonctionnel covariant fermé

Les deux morceaux sont maintenant :

\[
\mathcal T_{\rm alg}^{mn}
\]

calculé explicitement à partir de \(Q,J_0,U_0,P,X\), et :

\[
\mathcal K_\Gamma^{mn}[N]
\]

issu de la variation de connexion.

Ainsi :

\[
\boxed{
\frac{\delta I_N^{\rm loc}}{\delta h_{mn}}
=
\sqrt h
\left[
N\mathcal T_{\rm alg}^{mn}
+
\mathcal K_\Gamma^{mn}[N]
\right]
}
\]

où :

\[
I_N^{\rm loc}
=
\int d^3x\,
\sqrt h\,N C_N^{\rm loc}.
\]

Cette formule est un opérateur fonctionnel covariant : le calcul en coordonnées normales fournit les composantes ponctuelles de \(\mathcal T_{\rm alg}^{mn}\), tandis que \(\mathcal K_\Gamma^{mn}\) restaure exactement la dépendance différentielle de la connexion.


In [ ]:

FUNCTIONAL_METRIC_JET = {
    "algebraic_tensor":
        "T_alg^{mn} = explicit SymPy 3x3 symmetric response",

    "connection_tensor":
        (
            "K_Gamma^{mn}[N] = 1/4 D_i[N(A^{im}v^n+A^{in}v^m)] "
            "+ 1/4 D_j[N(A^{mj}v^n+A^{nj}v^m)] "
            "- 1/4 D_l[N(A^{mn}+A^{nm})v^l]"
        ),

    "full":
        (
            "delta I_N^loc / delta h_mn = "
            "sqrt(h) [N T_alg^{mn} + K_Gamma^{mn}[N]]"
        ),
}

CNLOC_COVARIANT_METRIC_FUNCTIONAL_JET_EXPLICIT = all([
    ALGEBRAIC_METRIC_RESPONSE_TENSOR_EXPLICIT,
    QJET_RESPONSE_EXPLICIT,
    CONNECTION_VARIATION_IBP_IDENTITY_VERIFIED,
])

assert CNLOC_COVARIANT_METRIC_FUNCTIONAL_JET_EXPLICIT

print(
    "CNLOC_COVARIANT_METRIC_FUNCTIONAL_JET_EXPLICIT =",
    CNLOC_COVARIANT_METRIC_FUNCTIONAL_JET_EXPLICIT
)



# 12 — Contrôle indépendant de l'identité de Legendre avec \(q_{ij}\)

Le signe dans :

\[
\mathcal A^{ij}
=
J_{,q_{ij}}^TX
+\frac12X^TQ_{,q_{ij}}X
+U_{,q_{ij}}
\]

est critique.

On effectue donc un témoin exact où \(Q\) est inversible, puis on compare :

1. la dérivée directe de
   \[
   C=-\frac12(P-J)^TQ^{-1}(P-J)+U;
   \]
2. la formule compacte en \(X=Q^{-1}(P-J)\).

Ce témoin ne sert pas à conclure physiquement ; il vérifie seulement l'algèbre du moteur.


In [ ]:

witness = {
    c1:sp.Rational(2,5),
    c2:sp.Rational(-1,7),
    c3:sp.Rational(1,4),
    c4:sp.Rational(3,10),
    s:sp.Rational(6,5),
    v[0]:sp.Rational(1,5),
    v[1]:sp.Rational(-1,6),
    v[2]:sp.Rational(1,7),

    g[0]:sp.Rational(1,9),
    g[1]:sp.Rational(-1,10),
    g[2]:sp.Rational(1,11),
}

for idx,qij in enumerate(list(q)):
    witness[qij] = sp.Rational(idx-4,17)

P_w = sp.Matrix([
    sp.Rational(i-3,13)
    for i in range(10)
])

Q_w = Q0.subs(witness)
J_w = J0.subs(witness)
U_w = U0.subs(witness)

assert Q_w.det() != 0

X_w = sp.simplify(
    Q_w.inv()*(P_w-J_w)
)

qtest = q[0,0]

# Keep q11 symbolic while all other witness data are fixed.
subs_except_q11 = {
    k:v0
    for k,v0 in witness.items()
    if k != qtest
}

Q_t = Q0.subs(subs_except_q11)
J_t = J0.subs(subs_except_q11)
U_t = U0.subs(subs_except_q11)

C_direct_t = sp.simplify(
    -sp.Rational(1,2)*
    (
        (P_w-J_t).T
        *Q_t.inv()
        *(P_w-J_t)
    )[0]
    +U_t
)

direct_q_derivative = sp.simplify(
    sp.diff(
        C_direct_t,
        qtest
    ).subs(
        qtest,
        witness[qtest]
    )
)

compact_q_derivative = sp.simplify(
    Aq[0,0]
    .subs(witness)
    .subs({
        X[i]:X_w[i]
        for i in range(10)
    })
)

assert sp.simplify(
    direct_q_derivative
    -
    compact_q_derivative
) == 0

LEGENDRE_QJET_WITNESS_PASS = True

print(
    "LEGENDRE_QJET_WITNESS_PASS =",
    LEGENDRE_QJET_WITNESS_PASS
)



# 13 — Contrôle du terme de normalisation \(P_A/\sqrt h\)

Un modèle jouet vérifie indépendamment que, lorsque :

\[
P=\Pi/\sqrt h,
\]

la variation à \(\Pi\) fixé contient bien :

\[
-X\,\delta P
=
+\frac12XP\,h^{mn}\delta h_{mn}.
\]


In [ ]:

u = sp.symbols("u", real=True)
Pi = sp.symbols("Pi", real=True)
qtoy = sp.symbols("qtoy", real=True)

# sqrt(h) = 1 + eps*u/2 at first order
sqrt_h_toy = 1+eps*u/2
P_toy = Pi/sqrt_h_toy

Q_toy = sp.Integer(3)
J_toy = qtoy
U_toy = qtoy**2

C_toy = (
    -sp.Rational(1,2)*
    (P_toy-J_toy)**2/Q_toy
    +U_toy
)

density_toy = sp.expand(
    sqrt_h_toy*C_toy
)

direct_toy = sp.simplify(
    sp.diff(
        density_toy,
        eps
    ).subs(eps,0)
)

P0_toy = Pi
X0_toy = (P0_toy-J_toy)/Q_toy
C0_toy = (
    -sp.Rational(1,2)*
    (P0_toy-J_toy)*X0_toy
    +U_toy
)

predicted_toy = sp.simplify(
    u/2*C0_toy
    +
    u/2*X0_toy*P0_toy
)

assert sp.simplify(
    direct_toy-predicted_toy
) == 0

NORMALIZED_MOMENT_DENSITY_VARIATION_WITNESS_PASS = True

print(
    "NORMALIZED_MOMENT_DENSITY_VARIATION_WITNESS_PASS =",
    NORMALIZED_MOMENT_DENSITY_VARIATION_WITNESS_PASS
)



# 14 — Ce que cette étape ne calcule pas

Cette étape ne forme volontairement pas encore :

\[
\{H[N],H[M]\}.
\]

Elle ne soustrait pas :

\[
D[\beta].
\]

Elle ne réduit pas encore un \(R_{HH}\) modulo :

\[
(p_\lambda,\chi,\psi,\rho).
\]

Elle ne forme pas encore la correction de Dirac.

Ces opérations appartiennent au **réassemblage HH immédiat**, avec le nouveau jet métrique désormais disponible.

Donc :

\[
\boxed{
\texttt{FULL\_HH\_CANONICAL\_BRACKET\_COMPUTED=False}
}
\]

reste intentionnel à la sortie de `.1.6.1`.


In [ ]:

FULL_HH_CANONICAL_BRACKET_COMPUTED = False
FULL_HH_DIRAC_BRACKET_COMPUTED = False
RHH_PHYSICAL_CLASSIFIED = False
HYPERSURFACE_ALGEBRA_CLOSED = False
DISPERSION_READY = False

assert not FULL_HH_CANONICAL_BRACKET_COMPUTED
assert not FULL_HH_DIRAC_BRACKET_COMPUTED
assert not RHH_PHYSICAL_CLASSIFIED
assert not HYPERSURFACE_ALGEBRA_CLOSED
assert not DISPERSION_READY

print(
    "FULL_HH_CANONICAL_BRACKET_COMPUTED =",
    FULL_HH_CANONICAL_BRACKET_COMPUTED
)
print(
    "DISPERSION_READY =",
    DISPERSION_READY
)



# 15 — Gates de `.1.6.1`


In [ ]:

GATES = {
    "normal_coordinate_first_variation_setup":
        NORMAL_COORDINATE_FIRST_VARIATION_SETUP,

    "covariant_local_invariants_reconstructed":
        COVARIANT_LOCAL_INVARIANTS_RECONSTRUCTED,

    "orthonormal_upstream_reduction_exact":
        ORTHONORMAL_UPSTREAM_REDUCTION_EXACT,

    "algebraic_metric_first_variation_explicit":
        ALGEBRAIC_METRIC_FIRST_VARIATION_EXPLICIT,

    "QJU_metric_response_explicit":
        QJU_METRIC_RESPONSE_EXPLICIT,

    "Legendre_compact_representation_ready":
        LEGENDRE_COMPACT_REPRESENTATION_READY,

    "normalized_moment_metric_response_explicit":
        NORMALIZED_MOMENT_METRIC_RESPONSE_EXPLICIT,

    "algebraic_metric_response_tensor_explicit":
        ALGEBRAIC_METRIC_RESPONSE_TENSOR_EXPLICIT,

    "qjet_response_explicit":
        QJET_RESPONSE_EXPLICIT,

    "connection_variation_IBP_identity_verified":
        CONNECTION_VARIATION_IBP_IDENTITY_VERIFIED,

    "Legendre_qjet_witness_pass":
        LEGENDRE_QJET_WITNESS_PASS,

    "normalized_moment_density_variation_witness_pass":
        NORMALIZED_MOMENT_DENSITY_VARIATION_WITNESS_PASS,

    "CNloc_covariant_metric_functional_jet_explicit":
        CNLOC_COVARIANT_METRIC_FUNCTIONAL_JET_EXPLICIT,

    "full_HH_canonical_bracket_computed":
        FULL_HH_CANONICAL_BRACKET_COMPUTED,

    "full_HH_Dirac_bracket_computed":
        FULL_HH_DIRAC_BRACKET_COMPUTED,

    "RHH_physical_classified":
        RHH_PHYSICAL_CLASSIFIED,

    "hypersurface_algebra_closed":
        HYPERSURFACE_ALGEBRA_CLOSED,

    "dispersion_ready":
        DISPERSION_READY,
}

for k,vv in GATES.items():
    print(k,":",vv)



# 16 — Verdict autorisé

Si tous les gates de dérivation passent :

\[
\boxed{
\texttt{
PASS-CNLOC-COVARIANT-METRIC-FUNCTIONAL-JET-
AND-CONNECTION-VARIATION-CLOSURE
}
}
\]

Le blocage de `.1.6` :

\[
\texttt{
BLOCKED-MISSING-CNLOC-COVARIANT-METRIC-FUNCTIONAL-JET
}
\]

peut alors être levé.

Mais le statut HH devient seulement :

\[
\boxed{
\texttt{HH-REASSEMBLY-NOW-AUTHORIZED}
}
\]

et non :

\[
\texttt{HYPERSURFACE-ALGEBRA-CLOSED}.
\]


In [ ]:

REQUIRED_PASS = [
    "normal_coordinate_first_variation_setup",
    "covariant_local_invariants_reconstructed",
    "orthonormal_upstream_reduction_exact",
    "algebraic_metric_first_variation_explicit",
    "QJU_metric_response_explicit",
    "Legendre_compact_representation_ready",
    "normalized_moment_metric_response_explicit",
    "algebraic_metric_response_tensor_explicit",
    "qjet_response_explicit",
    "connection_variation_IBP_identity_verified",
    "Legendre_qjet_witness_pass",
    "normalized_moment_density_variation_witness_pass",
    "CNloc_covariant_metric_functional_jet_explicit",
]

ALL_REQUIRED_PASS = all(
    GATES[k]
    for k in REQUIRED_PASS
)

if ALL_REQUIRED_PASS:
    FINAL_STATUS = (
        "PASS-CNLOC-COVARIANT-METRIC-FUNCTIONAL-JET-"
        "AND-CONNECTION-VARIATION-CLOSURE"
    )
    HH_NEXT_STATUS = "HH-REASSEMBLY-NOW-AUTHORIZED"
else:
    FINAL_STATUS = (
        "BLOCKED-CNLOC-COVARIANT-METRIC-JET-DERIVATION-FAIL"
    )
    HH_NEXT_STATUS = "HH-REASSEMBLY-NOT-AUTHORIZED"

assert ALL_REQUIRED_PASS

print("FINAL_STATUS =",FINAL_STATUS)
print("HH_NEXT_STATUS =",HH_NEXT_STATUS)
print("DISPERSION_READY =",DISPERSION_READY)



# 17 — Export machine-readable

Les expressions \(\mathcal T_{\rm alg}^{mn}\) et \(\mathcal A^{ij}\) sont exportées afin que le notebook HH suivant les réutilise **sans les réinventer**.


In [ ]:

artifact = {
    "notebook":
        "GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.1.6.1",

    "traceability":
        "REDERIVED_ACTUAL_GVH_QJU_NO_MODEL_MODIFICATION",

    "functional_jet":
        {
            "formula":
                (
                    "delta I_N^loc/delta h_mn = "
                    "sqrt(h) [N*T_alg^{mn} + K_Gamma^{mn}[N]]"
                ),

            "T_alg_components":
                {
                    f"{i}{j}":
                        str(Talg[i,j])
                    for i in range(3)
                    for j in range(i,3)
                },

            "A_q_components":
                {
                    f"{i}{j}":
                        str(Aq[i,j])
                    for i in range(3)
                    for j in range(3)
                },

            "K_Gamma":
                (
                    "1/4 D_i[N(A^{im}v^n+A^{in}v^m)] "
                    "+1/4 D_j[N(A^{mj}v^n+A^{nj}v^m)] "
                    "-1/4 D_l[N(A^{mn}+A^{nm})v^l]"
                ),

            "Legendre_relation":
                "Q X = P-J0",
        },

    "gates":
        GATES,

    "final_status":
        FINAL_STATUS,

    "HH_next_status":
        HH_NEXT_STATUS,

    "full_HH_canonical_bracket_computed":
        False,

    "full_HH_Dirac_bracket_computed":
        False,

    "RHH_physical_classified":
        False,

    "hypersurface_algebra_closed":
        False,

    "dispersion_ready":
        False,

    "next":
        (
            "GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.1.6.2_"
            "Strict_HH_Reassembly_with_Covariant_CNloc_Jet_"
            "Canonical_and_Dirac_Residual_Classification.ipynb"
        ),
}

export_dir = (
    Path("/content/gvh_exports")
    if Path("/content").exists()
    else Path.cwd()/"gvh_exports"
)

export_dir.mkdir(
    parents=True,
    exist_ok=True
)

artifact_path = export_dir / (
    "gvh_0.3.2.7.3.7.3.3.1.6.1_"
    "CNloc_covariant_metric_functional_jet.json"
)

artifact_path.write_text(
    json.dumps(
        artifact,
        indent=2
    ),
    encoding="utf-8"
)

print("Artifact:",artifact_path)



# Conclusion

Le bloc découvert par `.1.6` est maintenant isolé sous une forme calculable sans développer brutalement \(Q^{-1}\).

La structure finale à tester à l'exécution est :

\[
\boxed{
\frac{\delta}{\delta h_{mn}}
\int d^3x\,\sqrt h\,N C_N^{\rm loc}
=
\sqrt h
\left[
N\mathcal T_{\rm alg}^{mn}
+
\mathcal K_\Gamma^{mn}[N]
\right].
}
\]

Le terme :

\[
\mathcal T_{\rm alg}^{mn}
\]

est dérivé des variations métriques covariantes des invariants GVH, du secteur EH cinétique, de la mesure \(\sqrt h\) et des moments normalisés.

Le terme :

\[
\mathcal K_\Gamma^{mn}[N]
\]

restaure exactement la variation :

\[
\delta(D_i v_j)
=
-\delta\Gamma^k{}_{ij}v_k
\]

à \(v_i\) fixé.

Si les gates passent, aucune nouvelle branche intermédiaire n'est autorisée.

La suite est immédiatement :

\[
\boxed{
\mathbf{0.3.2.7.3.7.3.3.1.6.2}
}
\]

pour réassembler :

\[
\{H[N],H[M]\}_{\rm can},
\]

puis :

\[
R_{HH}^{\rm can},
\qquad
R_{HH}^{D},
\]

et enfin appliquer la classification préparée dans `.1.5`.

Jusqu'à ce réassemblage :

\[
\boxed{
\texttt{HYPERSURFACE\_ALGEBRA\_CLOSED=False},
\qquad
\mathrm{DISPERSION\_READY=False}.
}
\]
